# System-level simulation

> Many links, no bits: a PHY abstraction (SNR to packet error rate), power control, and the channel built on them.

In [ ]:
#| default_exp system

In [ ]:
#| hide
from nbdev.showdoc import *

System-level simulation in the sense of [Sionna SYS](https://nvlabs.github.io/sionna/sys/index.html): what matters
is how many links work and how well, not the bits of each transmission. A *PHY abstraction* replaces the bits. It
maps a transmission's SNR to a packet error rate, and the packet is lost with that probability. It is fast, so
it suits training and large numbers of agents. `LinkLevelChannel` (see `link`) simulates the bits instead, and
gives the same packet error rates (tested there).

The decisions of the system also belong here, since they need no bits: the transmit power and whether to transmit
at all (`PowerControl`). Which links transmit when is decided by the `Protocol`; links that transmit at the same time interfere
(`InterferenceBackend`).

`RadioChannel` is what channels at both levels share. It computes the *link budget* of a transmission: `h` from a
`Fading`, the power `P` from a `PowerControl`, and the SNR `P |h|² / noise_power`.

In [ ]:
#| export
from __future__ import annotations

import math
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np

from comm_core.backend import CommunicationBackend
from comm_core.channel import Channel
from comm_core.core import Message, Transmission, TransmissionResult
from comm_core.propagation import Fading, NoFading
from comm_core.topology import Link

In [ ]:
#| export
def _is_torch(x) -> bool:
    return type(x).__module__.partition('.')[0] == 'torch'


def _full(like, value, boolean: bool = False):
    "An array (or tensor) shaped like `like`, filled with `value`."
    if _is_torch(like):
        import torch
        return torch.full_like(like, value, dtype=torch.bool if boolean else None)
    return np.full(np.shape(like), value, dtype=bool if boolean else float)

## PHY abstraction

The packet error rate of an SNR. `'bpsk'` uses BPSK's bit error rate over AWGN, `Q(√(2 SNR))`, with independent
bit errors, so bigger packets fail more often. `'threshold'` receives a packet when the SNR is at or above a
threshold.

In [ ]:
#| export
def bpsk_bit_error_rate(
    snr: float, # Linear signal-to-noise ratio
) -> float:
    "Bit error rate of BPSK over AWGN: Q(sqrt(2 snr))."
    return 0.5 * math.erfc(math.sqrt(max(snr, 0.0)))


def packet_error_rate(
    snr_db: float, # Signal-to-noise ratio (dB); -inf: no signal
    size_bits: int, # Packet size
    model: str = 'bpsk', # 'bpsk' (independent BPSK bit errors) or 'threshold'
    threshold_db: float = 10.0, # 'threshold': packets at or above this SNR are received
) -> float:
    "Probability that a packet is lost."
    if snr_db == math.inf:
        return 0.0
    if snr_db == -math.inf:
        return 1.0
    if model == 'threshold':
        return 0.0 if snr_db >= threshold_db else 1.0
    if model == 'bpsk':
        ber = bpsk_bit_error_rate(10 ** (snr_db / 10))
        return 1.0 - (1.0 - ber) ** max(int(size_bits), 1)
    raise ValueError(f"model: 'bpsk' or 'threshold', not {model!r}")

In [ ]:
assert packet_error_rate(30, 1000) < 1e-6 and packet_error_rate(-5, 1000) > 0.99
assert packet_error_rate(4, 10) < packet_error_rate(4, 1000)                  # bigger packets fail more often
assert packet_error_rate(9, 100, 'threshold') == 1.0 and packet_error_rate(10, 100, 'threshold') == 0.0
assert packet_error_rate(-math.inf, 8) == 1.0 and packet_error_rate(math.inf, 8) == 0.0

## Power control

A `PowerControl` decides the transmit power `P` from the channel gain `|h|²`, and whether to transmit at all. A
link it does not use is an *outage*: nothing is sent, so the receiver gets nothing.

* `FixedPower` uses the same power always, by default the link's `tx_power_w`.
* `ChannelInversion` uses just enough power to reach a target SNR: `P = γ σ² / |h|²`, where `γ` is the target SNR
  and `σ²` the noise power. A link needing more than `max_power` is not used.
* `MessagePower` uses the power the sender chose, e.g. a learned power control, put in the message's metadata.

`allocate` is vectorized (numpy arrays or torch tensors), so training code can handle a whole batch of links at once.

In [ ]:
#| export
class PowerControl(ABC):
    "How much power a transmission uses, and whether it is sent at all."

    @abstractmethod
    def allocate(
        self,
        gain, # Channel power gain |h|^2 (a number, an array or a tensor)
        noise_power, # Noise power at the receiver
        link: Link | None = None, # The link (for one transmission)
        message: Message | None = None, # The message (for one transmission)
    ):
        "(power, transmit): the transmit power, and whether to transmit (False: outage), shaped like `gain`."


class FixedPower(PowerControl):
    "The same power always."

    def __init__(
        self,
        power: float | None = None, # Transmit power; None: the link's `tx_power_w` (1 without a link)
    ):
        self.power = power

    def allocate(self, gain, noise_power, link=None, message=None):
        p = self.power if self.power is not None else (link.tx_power_w if link is not None else 1.0)
        return _full(gain, p), _full(gain, True, boolean=True)


class ChannelInversion(PowerControl):
    "Just enough power to reach `target_snr_db`: P = snr noise_power / |h|^2; links needing more than `max_power` are not used."

    def __init__(
        self,
        target_snr_db: float = 10.0, # SNR at the receiver
        max_power: float = math.inf, # Power budget
    ):
        self.target_snr_db, self.max_power = target_snr_db, max_power

    def allocate(self, gain, noise_power, link=None, message=None):
        if not (_is_torch(gain) or isinstance(gain, np.ndarray)):
            gain = np.float64(gain)
        with np.errstate(divide='ignore'):                 # |h|^2 = 0: infinite power, an outage
            power = 10 ** (self.target_snr_db / 10) * noise_power / gain
        return power, power <= self.max_power


class MessagePower(PowerControl):
    "The power the sender chose (e.g. a learned power control), in `message.metadata[key]`; no power or 0: not sent."

    def __init__(
        self,
        key: str = 'tx_power', # Metadata key of the power
        max_power: float = math.inf, # Larger powers are clipped to it
    ):
        self.key, self.max_power = key, max_power

    def allocate(self, gain, noise_power, link=None, message=None):
        if message is None:
            raise ValueError("MessagePower reads the power from the message")
        p = min(float(message.metadata.get(self.key, 0.0)), self.max_power)
        return p, p > 0

In [ ]:
show_doc(PowerControl.allocate)

In [ ]:
p, ok = ChannelInversion(target_snr_db=10, max_power=10).allocate(np.array([1.0, 0.5, 0.01, 0.0]), noise_power=1.0)
assert np.allclose(p[:3], [10, 20, 1000]) and p[3] == np.inf and ok.tolist() == [True, False, False, False]
p, ok = FixedPower().allocate(0.3, 1.0, link=Link('a', 'b', tx_power_w=0.2))
assert float(p) == 0.2 and bool(ok)
assert MessagePower(max_power=5).allocate(1.0, 1.0, message=Message('a', 'b', None, metadata={'tx_power': 9})) == (5.0, True)
assert MessagePower().allocate(1.0, 1.0, message=Message('a', 'b', None)) == (0.0, False)

## Radio channels

The link budget of a transmission, and what both levels share: a `Fading`, a `PowerControl`, the noise power and
a random generator. Subclasses decide what arrives.

**The results** of both levels report the SNR, the energy (power × duration) and the latency (symbols / bandwidth,
0 with an infinite bandwidth). Their `metadata` holds `channel_gain_db` and `tx_power`, and `outage` for links the
power control did not use (`success=False`, 0 bits sent).

A transmission can be given the power of other transmissions at its receiver (`interference`, see
`InterferenceBackend`). Its `snr_db` is then the SINR, `P |h|² / (noise_power + interference)`, and the
metadata has `interference_power`.

In [ ]:
#| export
def _child_seed(seed: int | None, i: int) -> int | None:
    return None if seed is None else int(np.random.SeedSequence([seed, i]).generate_state(1)[0])


@dataclass
class LinkBudget:
    "A transmission's channel coefficient, transmit power and SNR."
    h: complex # Channel coefficient (math.inf: the nodes are at the same place)
    power: float # Transmit power
    transmit: bool # False: an outage, nothing is sent
    snr_db: float # P |h|^2 / noise power, dB

    @property
    def gain_db(self) -> float:
        "|h|^2 in dB."
        gain = abs(self.h) ** 2
        return 10 * math.log10(gain) if gain > 0 else -math.inf


class RadioChannel(Channel):
    "A channel over radio links: h from a fading, P from a power control, noise; subclasses decide what arrives."

    def __init__(
        self,
        fading: Fading | None = None, # The coefficient h of a transmission; None: `NoFading()` (h = 1)
        power_control: PowerControl | None = None, # Transmit power and outages; None: `FixedPower()` (the link's tx_power_w)
        noise_power: float = 1.0, # Noise power at the receiver (same unit as the power)
        seed: int | None = None, # Seed of the channel's randomness and of the fading
    ):
        self.fading = fading if fading is not None else NoFading()
        self.power_control = power_control if power_control is not None else FixedPower()
        self.noise_power = noise_power
        self.rng = np.random.default_rng()
        self.reset(seed)

    def reset(self, seed: int | None = None) -> None:
        "`seed` reseeds the channel and the fading."
        if seed is not None:
            self.rng = np.random.default_rng(_child_seed(seed, 0))
        self.fading.reset(_child_seed(seed, 1))

    def set_positions(self, positions) -> None:
        "Where the nodes are, for a fading that depends on it (e.g. `RayTracedFading`)."
        if hasattr(self.fading, 'set_positions'):
            self.fading.set_positions(positions)

    def link_budget(
        self,
        transmission: Transmission,
        link: Link,
    ) -> LinkBudget:
        "The coefficient, power and SNR of a transmission (nodes at the same place: a perfect link)."
        h = complex(self.fading.coefficient(transmission, link))
        gain = abs(h) ** 2
        if math.isinf(gain):
            return LinkBudget(h, 0.0, link.enabled, math.inf)
        power, transmit = self.power_control.allocate(gain, self.noise_power, link=link, message=transmission.message)
        power = float(power)
        snr = power * gain / self.noise_power
        return LinkBudget(h, power, bool(transmit) and link.enabled, 10 * math.log10(snr) if snr > 0 else -math.inf)

    def _sinr_db(self, budget: LinkBudget, interference: float = 0.0) -> float:
        "P |h|^2 / (noise power + interference), dB: the SNR without interference."
        if budget.snr_db == math.inf or not interference:
            return budget.snr_db
        sinr = budget.power * abs(budget.h) ** 2 / (self.noise_power + interference)
        return 10 * math.log10(sinr) if sinr > 0 else -math.inf

    def _num_bits(self, message: Message) -> int:
        "Bits of a message on the air."
        return message.size_bits or 0

    def _num_symbols(self, num_bits: int) -> int:
        return num_bits

    def _lost(self, transmission: Transmission, link: Link, budget: LinkBudget, snr_db: float, **reason) -> TransmissionResult:
        "Sent (and paid for), but not received."
        bits = self._num_bits(transmission.message)
        latency, energy = self._timing(budget, self._num_symbols(bits), link)
        return TransmissionResult(success=False, transmission=transmission, payload=None, latency=latency, size_bits=bits,
                                  energy=energy, snr_db=snr_db, error_rate=1.0, metadata=self._metadata(budget, **reason))

    def _metadata(self, budget: LinkBudget, **extra) -> dict:
        return {'channel_gain_db': budget.gain_db, 'tx_power': budget.power if budget.transmit else 0.0, **extra}

    def _outage(self, transmission: Transmission, budget: LinkBudget) -> TransmissionResult:
        return TransmissionResult(success=False, transmission=transmission, payload=None, size_bits=0,
                                  snr_db=budget.snr_db, metadata=self._metadata(budget, outage=True))

    @staticmethod
    def _timing(budget: LinkBudget, num_symbols: int, link: Link) -> tuple:
        "(latency, energy) of `num_symbols` symbols."
        duration = num_symbols / link.bandwidth_hz if math.isfinite(link.bandwidth_hz) else 0.0
        return duration, budget.power * (duration if duration else num_symbols)

In [ ]:
show_doc(RadioChannel.link_budget)

## System-level channel

The SNR of a transmission's link budget gives its packet error rate, and the packet is lost with that probability.
A packet that arrives is the payload as sent. `message.size_bits` is the packet size, and the timing assumes one bit
per symbol.

In [ ]:
#| export
class SystemLevelChannel(RadioChannel):
    "A PHY abstraction: SNR -> packet error rate -> the packet arrives intact or is lost. No bits are simulated."

    def __init__(
        self,
        fading: Fading | None = None, # The coefficient h of a transmission; None: h = 1
        power_control: PowerControl | None = None, # Transmit power and outages; None: the link's tx_power_w
        noise_power: float = 1.0, # Noise power at the receiver
        per_model: str = 'bpsk', # Packet error rate of the SNR: 'bpsk' or 'threshold' (see `packet_error_rate`)
        snr_threshold_db: float = 10.0, # The 'threshold' model's SNR
        seed: int | None = None, # Seed of the packet losses and the fading
    ):
        self.per_model, self.snr_threshold_db = per_model, snr_threshold_db
        super().__init__(fading, power_control, noise_power, seed)

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
        interference: float = 0.0, # Power of the other transmissions at the receiver (see `InterferenceBackend`)
        budget: LinkBudget | None = None, # The transmission's link budget, if already computed
    ) -> TransmissionResult:

        if budget is None:
            budget = self.link_budget(transmission, link)
        if not budget.transmit:
            return self._outage(transmission, budget)
        sinr_db = self._sinr_db(budget, interference)
        message = transmission.message
        size_bits = message.size_bits or 0
        per = packet_error_rate(sinr_db, size_bits or 1, self.per_model, self.snr_threshold_db)
        success = bool(self.rng.random() >= per)
        latency, energy = self._timing(budget, size_bits, link)
        return TransmissionResult(
            success=success,
            transmission=transmission,
            payload=message.payload if success else None,
            latency=latency,
            size_bits=size_bits,
            energy=energy,
            snr_db=sinr_db,
            error_rate=per,
            metadata=self._metadata(budget, **({'interference_power': interference} if interference else {})),
        )

### Tests

In [ ]:
def send(ch, link=Link('a', 'b'), n=1, size_bits=100, **metadata):
    t = Transmission(message=Message('a', 'b', 'hi', size_bits=size_bits, metadata=metadata), sender='a', receiver='b', start_time=0.0)
    return [ch.transmit(t, link) for _ in range(n)]

for snr_db, delivered in ((10, 1.0), (0, 0.0)):                 # 100 bits: always through at 10 dB, never at 0 dB
    rs = send(SystemLevelChannel(noise_power=10 ** (-snr_db / 10), seed=0), n=50)
    assert np.mean([r.success for r in rs]) == delivered and abs(rs[0].snr_db - snr_db) < 1e-9
r = rs[0]
assert r.size_bits == 100 and r.energy == 100 and r.latency == 0 and r.metadata == {'channel_gain_db': 0.0, 'tx_power': 1.0}
# in between: lost as often as the packet error rate says, reproducibly
ch = SystemLevelChannel(noise_power=10 ** -0.6, seed=0)        # 6 dB, 100 bits
ch.reset(seed=3); first = [r.success for r in send(ch, n=2000)]
ch.reset(seed=3); assert [r.success for r in send(ch, n=2000)] == first
assert abs(1 - np.mean(first) - packet_error_rate(6, 100)) < 0.03
# the threshold model, channel inversion (outages), a perfect link, a finite bandwidth
assert send(SystemLevelChannel(noise_power=1.0, per_model='threshold', snr_threshold_db=-1))[0].success
for h, sent in ((1.2, True), (0.1, False)):
    r, = send(SystemLevelChannel(fading=NoFading(h), power_control=ChannelInversion(10, max_power=10)))
    assert r.success == sent and r.size_bits == (100 if sent else 0) and r.metadata.get('outage', False) != sent
r, = send(SystemLevelChannel(fading=NoFading(math.inf), noise_power=1e9))
assert r.success and r.snr_db == math.inf
r, = send(SystemLevelChannel(), link=Link('a', 'b', bandwidth_hz=1e6, tx_power_w=0.5))
assert abs(r.latency - 1e-4) < 1e-15 and abs(r.energy - 0.5e-4) < 1e-15

In a `Network`, the topology and the protocol decide which transmissions happen, and the channel decides what arrives:

In [ ]:
from comm_core import Network, Node, FullMeshTopology, IdealProtocol, AnalyticalBackend
from comm_core.propagation import RayleighFading

In [ ]:
ch = SystemLevelChannel(fading=RayleighFading(), power_control=ChannelInversion(10, max_power=20), seed=0)
nodes = ['a', 'b', 'c']
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), IdealProtocol(), AnalyticalBackend(ch))
net.reset(seed=0)
for step in range(20):
    for s in nodes:
        for r in nodes:
            if s != r:
                net.send(Message(s, r, payload=step, size_bits=64))
    net.step()
m = net.metrics.summary()
assert m.transmissions == 20 * 6 and 0 < m.failed < m.transmissions        # deep fades: outages

## Interference

`InterferenceBackend` makes the transmissions of a `Network` step share one medium: they happen at the same time and
interfere. Each transmission's SINR counts the power the other senders put at its receiver,

`SINR = P |h|² / (noise_power + Σ P' |h'|²)`,

where `P'` is another sender's power and `h'` the coefficient from it to the receiver, from the same fading. The
interference is treated as Gaussian noise. Then:

* **Half-duplex** (the default): a node that transmits in a step cannot receive in it.
* **One radio per node**: a sender's transmissions in a step don't interfere with each other, and it interferes
  with others at its strongest transmit power.
* **Power control** is open-loop: `ChannelInversion` targets the SNR, not the SINR.

It works with both levels: `SystemLevelChannel` uses the SINR for the packet error rate, and `LinkLevelChannel` adds
the interference to the noise of every bit.

Whether to avoid simultaneous transmissions is the `Protocol`'s job: `TDMAProtocol` gives each node its own
slots, and `SlottedALOHA` transmits at random (see `protocol`). With `IdealProtocol`, everything sent before a step goes out in that step, so if every node talks at once,
nobody hears (half-duplex).

In [ ]:
#| export
class InterferenceBackend(CommunicationBackend):
    "The transmissions of a step share the medium: each one's SINR counts the power of the other senders at its receiver."

    def __init__(
        self,
        channel: RadioChannel, # System or link level; its fading also gives the interfering paths
        half_duplex: bool = True, # A node that transmits in a step cannot receive in it
    ):
        self.channel, self.half_duplex = channel, half_duplex

    def transmit(self, transmission: Transmission, link: Link) -> TransmissionResult:
        "A transmission alone: no interference."
        return self.channel.transmit(transmission, link)

    def transmit_all(
        self,
        transmissions: List[Tuple[Transmission, Link]], # The transmissions of one step, with their links
    ) -> List[TransmissionResult]:
        "The transmissions of one step, at the same time."
        budgets = [self.channel.link_budget(t, link) for t, link in transmissions]
        emitted = {}                                   # sender -> its power on the air (its strongest transmission)
        for (t, _), budget in zip(transmissions, budgets):
            if budget.transmit:
                emitted[t.sender] = max(emitted.get(t.sender, 0.0), budget.power)
        gains = {}                                     # |h|^2 of the interfering paths, once per step

        def gain(sender, receiver, message):
            if (sender, receiver) not in gains:
                path = Transmission(message=message, sender=sender, receiver=receiver, start_time=0.0)
                gains[sender, receiver] = abs(complex(self.channel.fading.coefficient(path, Link(sender, receiver)))) ** 2
            return gains[sender, receiver]

        results = []
        for (t, link), budget in zip(transmissions, budgets):
            if budget.transmit and self.half_duplex and t.receiver in emitted:
                results.append(self.channel._lost(t, link, budget, budget.snr_db, half_duplex=True))
                continue
            interference = sum(p * gain(s, t.receiver, t.message) for s, p in emitted.items()
                               if p > 0 and s not in (t.sender, t.receiver))
            results.append(self.channel.transmit(t, link, interference=interference, budget=budget))
        return results

    def reset(self, seed: int | None = None) -> None:
        self.channel.reset(seed)

    def step(self, time: float) -> None:
        self.channel.update(time)

In [ ]:
show_doc(InterferenceBackend.transmit_all)

### Tests

In [ ]:
from comm_core.propagation import RayTracedFading

In [ ]:
# 21 positions on a line, |H|^2 = 1e-6 / d^3 (d in cells)
xs = np.arange(21)
d = np.abs(np.subtract.outer(xs, xs)).astype(float)
H = np.where(d > 0, 1e-3 / np.maximum(d, 1) ** 1.5, 0)
table = RayTracedFading(H, [(x, 0) for x in xs])
ch = SystemLevelChannel(fading=table, noise_power=1e-10, per_model='threshold', snr_threshold_db=10, seed=0)
backend = InterferenceBackend(ch)
def tx(s, r, bits=64):
    return Transmission(message=Message(s, r, 'hi', size_bits=bits), sender=s, receiver=r, start_time=0.0), Link(s, r)

table.set_positions({'a': (0, 0), 'b': (1, 0), 'c': (20, 0), 'd': (19, 0)})
alone, = backend.transmit_all([tx('a', 'b')])
assert alone.success and abs(alone.snr_db - 40) < 1e-9                 # 1 W x 1e-6 / 1e-10: 40 dB
# c -> d far away: a little interference at b
far = backend.transmit_all([tx('a', 'b'), tx('c', 'd')])
i = 1e-6 / 19 ** 3
assert all(r.success for r in far) and abs(far[0].metadata['interference_power'] - i) < 1e-20
assert abs(far[0].snr_db - 10 * np.log10(1e-6 / (1e-10 + i))) < 1e-9
# c -> d next to b: a-b drops to 9 dB, below the threshold; c-d (a is 4 cells from d) still gets through
table.set_positions({'c': (3, 0), 'd': (4, 0)})
near = backend.transmit_all([tx('a', 'b'), tx('c', 'd')])
assert not near[0].success and 8.9 < near[0].snr_db < 9.1 and near[1].success
# half-duplex: a and b talk to each other at once, neither hears
both = backend.transmit_all([tx('a', 'b'), tx('b', 'a')])
assert not any(r.success for r in both) and all(r.metadata['half_duplex'] for r in both) and both[0].size_bits == 64
assert all(r.success for r in InterferenceBackend(ch, half_duplex=False).transmit_all([tx('a', 'b'), tx('b', 'a')]))
# one sender, two receivers: no interference with itself
two = backend.transmit_all([tx('a', 'b'), tx('a', 'c')])
assert all(r.success for r in two) and 'interference_power' not in two[0].metadata
# outages are not on the air: they don't interfere
ch_inv = SystemLevelChannel(fading=table, power_control=ChannelInversion(20, max_power=1e-6), noise_power=1e-10)
r = InterferenceBackend(ch_inv).transmit_all([tx('a', 'b'), tx('c', 'd')])
assert all(x.metadata.get('outage') for x in r)                        # each needs 1e-2 W > 1e-6 W

In a `Network`, a step's transmissions are the ones that happen at the same time:

In [ ]:
from comm_core import Network, Node, FullMeshTopology, IdealProtocol

In [ ]:
nodes = ['a', 'b', 'c', 'd']
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), IdealProtocol(), InterferenceBackend(ch))
table.set_positions({'a': (0, 0), 'b': (1, 0), 'c': (20, 0), 'd': (19, 0)})
net.send(Message('a', 'b', 'x', size_bits=64))
net.send(Message('c', 'd', 'y', size_bits=64))
assert [r.success for r in net.step()] == [True, True] and net.receive('b')[0].payload == 'x'
# everybody talks at once: every node is transmitting, so nobody hears
for s in nodes:
    for r in nodes:
        if s != r:
            net.send(Message(s, r, 'z', size_bits=64))
assert not any(r.success for r in net.step())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()